# RE:ARCHIVE — 무료 Real-ESRGAN 애니 원화/파츠 고화질 업스케일

원본 이미지는 그대로 ZIP에 보존하고, 별도 `upscaled/`에 저장합니다. AI가 얼굴이나 가려진 파츠를 자동 창작하는 단계가 아닙니다. 무료 GPU Colab에서만 실행합니다.

In [ ]:
from pathlib import Path
import zipfile,subprocess,sys,uuid,json,hashlib
from PIL import Image
from google.colab import files
BASE=Path("/content")/("rearchive_upscale_"+uuid.uuid4().hex[:8])
BASE.mkdir()
GPU=bool(subprocess.run(["bash","-lc","nvidia-smi >/dev/null"],capture_output=True).returncode==0)
if not GPU:raise RuntimeError("Real-ESRGAN에는 Colab GPU 런타임이 필요합니다")
u=files.upload()
if len(u)!=1:raise RuntimeError("REARCHIVE_ART_SOURCE_INPUT.zip 한 개를 업로드하세요")
fn,raw=next(iter(u.items()))
if not fn.lower().endswith(".zip"):raise RuntimeError("ZIP만 허용")
zpath=BASE/"source.zip";zpath.write_bytes(raw)
src=BASE/"source";src.mkdir()
with zipfile.ZipFile(zpath) as z:
    entries=[x for x in z.infolist() if not x.is_dir()]
    if len(entries)>350 or sum(x.file_size for x in entries)>700_000_000:raise RuntimeError("ZIP size limit")
    for i in entries:
        from pathlib import PurePosixPath
        p=PurePosixPath(i.filename)
        if p.is_absolute() or ".." in p.parts or "\\" in i.filename or i.file_size>90_000_000:raise RuntimeError("Unsafe ZIP")
        dest=src.joinpath(*p.parts);dest.parent.mkdir(parents=True,exist_ok=True);dest.write_bytes(z.read(i))
print("Input PNG files:",len(list(src.rglob("*.png"))))

In [ ]:
#@title AI 모델 준비
repo=BASE/"Real-ESRGAN"
subprocess.run(["git","clone","--depth","1","https://github.com/xinntao/Real-ESRGAN.git",str(repo)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","basicsr","facexlib","gfpgan","realesrgan"],check=True)
# Colab torch/torchvision mismatch may require a matching version; do not silently fall back to non-AI resize.
print("Ready: official Real-ESRGAN_x4plus_anime_6B, default tile=256")

In [ ]:
#@title 실제 2x/4x 업스케일 — 원본은 보존
SCALE=2 #@param [2,4]
SCOPE="references" #@param ["references","parts"]
if SCOPE=="parts":
    images=sorted((src/"parts").rglob("*.png"))
    if not images:raise RuntimeError("PARTS_NOT_AUTHORED: 독립 PNG 파츠가 없습니다")
else:
    images=[p for p in [src/"art/S01_front_neutral.png",src/"references/S02_left_45.png",src/"references/S03_right_45.png"] if p.is_file()]
    if not images:raise RuntimeError("NO_ART_MASTER")
runin=BASE/"input_imgs";runin.mkdir()
mapping={}
for idx,p in enumerate(images):
    with Image.open(p) as im:
        im.load()
        if im.mode!="RGBA" or im.size!=(1536,2304):raise RuntimeError("Expected original RGBA registered canvas: "+str(p))
    name=f"{idx:04d}.png";(runin/name).write_bytes(p.read_bytes());mapping[name]=p.relative_to(src).as_posix()
out=BASE/"AI_output";out.mkdir()
subprocess.run([sys.executable,str(repo/"inference_realesrgan.py"),"-n","RealESRGAN_x4plus_anime_6B","-i",str(runin),"-o",str(out),
                "-s",str(SCALE),"-t","256","--alpha_upsampler","bicubic","--suffix","ai","--ext","png"],cwd=repo,check=True)
results={}
for name,original in mapping.items():
    f=out/(Path(name).stem+"_ai.png")
    if not f.is_file():raise RuntimeError("AI_OUTPUT_MISSING:"+name)
    with Image.open(f) as im:
        im.load()
        if im.mode!="RGBA" or im.size!=(1536*SCALE,2304*SCALE):
            raise RuntimeError("AI_SIZE_OR_ALPHA_BAD:"+str(f))
    results[original]=f
print("AI-upscaled:",len(results),"files. Master original not overwritten.")

In [ ]:
#@title 원본 및 AI 결과 ZIP 다운로드
import zipfile
from google.colab import files
deliver=BASE/"REARCHIVE_AI_UPSCALED_WITH_ORIGINALS.zip"
status={"model":"RealESRGAN_x4plus_anime_6B","scale":SCALE,"scope":SCOPE,
        "upscaled_count":len(results),"originals_preserved":True,
        "created_underpaint":False,"rigged":False}
with zipfile.ZipFile(deliver,"w",zipfile.ZIP_DEFLATED,compresslevel=5) as z:
    for p in sorted(src.rglob("*")):
        if p.is_file():z.write(p,"originals/"+p.relative_to(src).as_posix())
    for original,f in results.items():z.write(f,"upscaled/"+original)
    z.writestr("UPSCALE_STATUS.json",json.dumps(status,ensure_ascii=False,indent=2))
    assert z.testzip() is None
files.download(str(deliver))